# 🏎️ Clasificación de Frenado con Red Neuronal (MLP) — Forza Motorsport
## Telemetría → ¿El conductor está frenando?

---

### ¿Qué problema resolvemos?

A partir de datos de telemetría del juego **Forza Motorsport**, queremos predecir si el conductor está **frenando (1)** o **no frenando (0)** en cada instante de la carrera.

Esto es un problema de **clasificación binaria**. El script original usaba regresión logística con `scipy.optimize`. Aquí lo rehacemos completamente con PyTorch, incorporando todo lo visto en los cuadernillos:

| Concepto | Script original | Este notebook |
|---|---|---|
| Modelo | Regresión logística manual | Red neuronal MLP |
| Optimización | `scipy.optimize.minimize` | `Adam` + `autograd` |
| Datos | Arrays NumPy directos | `Dataset` + `DataLoader` |
| Guardado | ❌ | `state_dict` + mejor modelo |
| GPU | ❌ | ✅ `.to(device)` |

### Arquitectura

```
Entrada (2)  →  Capa Oculta 1 (32, ReLU)  →  Capa Oculta 2 (16, ReLU)  →  Salida (1, Sigmoid)
 Velocidad
 RPM Motor                                                               →  P(frenando)
```

> **Dataset:** `telemetry-rio-5-laps.csv`  
> Sube el archivo CSV a tu entorno Colab antes de ejecutar.

## 📦 Paso 1 — Importaciones

Importamos todo lo que necesitaremos. Nota que ya no usamos `scipy.optimize` — PyTorch se encarga de la optimización completa.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo : {device}')
print(f'PyTorch     : {torch.__version__}')

Dispositivo : cuda
PyTorch     : 2.10.0+cu128


## 📂 Paso 2 — Carga y Análisis de Datos

El dataset de telemetría tiene más de 70 columnas. Nos quedamos con las que tienen sentido físico para predecir el frenado:

- **`speed`** → velocidad del vehículo (m/s)
- **`current_engine_rpm`** → revoluciones del motor

La variable objetivo **`brake`** tiene valores de 0–255. La convertimos a binaria:
- `brake > 0` → **1 (Frenando)**
- `brake == 0` → **0 (No frenando)**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
ruta_dataset = 'telemetry-rio-5-laps.csv'

try:
    data = pd.read_csv(ruta_dataset)
except FileNotFoundError:
    print(f"❌ Archivo no encontrado: '{ruta_dataset}'")
    raise

print('=' * 55)
print('        ANÁLISIS DEL DATASET DE TELEMETRÍA')
print('=' * 55)
print(f'  Filas x Columnas  : {data.shape}')
print(f'  Tipos de datos    :')
print(data.dtypes.value_counts().to_string(header=False))

# Limpieza: eliminamos filas con nulos en nuestras columnas
cols = ['speed', 'current_engine_rpm', 'brake']
antes = len(data)
data = data.dropna(subset=cols)
print(f'\n  Filas eliminadas (nulos): {antes - len(data)}')
print(f'  Filas limpias restantes : {len(data)}')
print('=' * 55)

In [ ]:
# Extraemos features y etiqueta binaria
X_raw = data[['speed', 'current_engine_rpm']].values.astype(np.float32)
y_raw = (data['brake'] > 0).astype(np.float32).values

print(f'Distribución de clases:')
unique, counts = np.unique(y_raw, return_counts=True)
for cls, cnt in zip(unique, counts):
    label = 'Frenando (1)   ' if cls == 1 else 'No frenando (0)'
    pct = cnt / len(y_raw) * 100
    print(f'  {label}: {cnt:>6} ejemplos  ({pct:.1f}%)')

print(f'\nTotal de ejemplos: {len(y_raw)}')

## 📊 Paso 3 — Visualización de los Datos

Antes de entrenar, visualizamos la distribución de los datos. Esto nos ayuda a entender si las clases son **separables** y qué tipo de frontera de decisión necesitamos.

In [ ]:
# Muestra aleatoria para visualizar (el dataset completo puede ser muy denso)
idx = np.random.choice(len(X_raw), size=min(1500, len(X_raw)), replace=False)
X_viz, y_viz = X_raw[idx], y_raw[idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: velocidad vs RPM coloreado por clase
pos = y_viz == 1
neg = y_viz == 0
axes[0].plot(X_viz[pos, 0], X_viz[pos, 1], 'k+', lw=1.5, ms=7, label='Frenando (1)', alpha=0.6)
axes[0].plot(X_viz[neg, 0], X_viz[neg, 1], 'ko', mfc='gold', ms=6, mec='k', mew=0.8, label='No frenando (0)', alpha=0.4)
axes[0].set_xlabel('Velocidad (m/s)', fontsize=11)
axes[0].set_ylabel('RPM Motor', fontsize=11)
axes[0].set_title('Distribución de Frenado', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Histograma de velocidad por clase
axes[1].hist(X_raw[y_raw == 1, 0], bins=40, alpha=0.6, color='crimson', label='Frenando')
axes[1].hist(X_raw[y_raw == 0, 0], bins=40, alpha=0.6, color='steelblue', label='No frenando')
axes[1].set_xlabel('Velocidad (m/s)', fontsize=11)
axes[1].set_ylabel('Frecuencia', fontsize=11)
axes[1].set_title('Distribución de Velocidad por Clase', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## ✂️ Paso 4 — Normalización y Split Train/Test

Dividimos los datos en **entrenamiento (80%)** y **prueba (20%)**. La normalización se calcula **solo sobre el train** y se aplica también al test — esto evita el *data leakage*.

> ⚠️ Guardamos `mu` y `sigma` del train. Los usaremos en la predicción de casos nuevos.

In [ ]:
# Split 80/20 estratificado (mantiene proporción de clases)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# Normalización z-score (calculada SOLO sobre train)
mu    = X_train.mean(axis=0)
sigma = X_train.std(axis=0)

X_train_norm = (X_train - mu) / sigma
X_test_norm  = (X_test  - mu) / sigma   # mismos mu/sigma del train

print(f'Train : {X_train_norm.shape[0]:>6} ejemplos')
print(f'Test  : {X_test_norm.shape[0]:>6} ejemplos')
print(f'\nNormalización (media, desv.std):')
print(f'  Velocidad : μ={mu[0]:.2f}, σ={sigma[0]:.2f}')
print(f'  RPM Motor : μ={mu[1]:.2f}, σ={sigma[1]:.2f}')

## 🗂️ Paso 5 — Dataset y DataLoader

Seguimos el patrón del cuadernillo `03_pytorch_datasets`: creamos una clase `Dataset` personalizada que hereda de `torch.utils.data.Dataset` e implementa los tres métodos obligatorios:

- `__init__` → almacena los datos como tensores
- `__len__` → devuelve cuántos ejemplos hay
- `__getitem__` → devuelve un ejemplo por índice

Luego `DataLoader` se encarga de crear los **mini-batches** automáticamente, con `shuffle=True` para mezclar en cada época.

In [ ]:
class TelemetriaDataset(Dataset):
    """Dataset de telemetría para clasificación de frenado."""

    def __init__(self, X: np.ndarray, y: np.ndarray):
        # Convertimos a tensores al inicializar (ya no hace falta hacerlo en el bucle)
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float().unsqueeze(1)  # shape: (m, 1)

    def __len__(self):
        # Obligatorio: cuántos ejemplos tiene el dataset
        return len(self.X)

    def __getitem__(self, idx):
        # Obligatorio: devuelve el par (features, etiqueta) del índice idx
        return self.X[idx], self.y[idx]


# Instanciamos datasets
train_dataset = TelemetriaDataset(X_train_norm, y_train)
test_dataset  = TelemetriaDataset(X_test_norm,  y_test)

# DataLoaders — batch_size=256 es un buen punto de partida para este tamaño de dataset
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

print(f'Train dataset : {len(train_dataset)} ejemplos  →  {len(train_loader)} batches de 256')
print(f'Test  dataset : {len(test_dataset)}  ejemplos  →  {len(test_loader)} batches de 256')

# Inspeccionamos un batch
x_batch, y_batch = next(iter(train_loader))
print(f'\nBatch de ejemplo → X: {x_batch.shape}  y: {y_batch.shape}')

## 🧠 Paso 6 — El Modelo MLP

Usamos el patrón `nn.Module` del cuadernillo `02_pytorch_nn` con **dos capas ocultas** y activación **ReLU**.

La capa de salida usa **Sigmoid** porque es clasificación binaria — necesitamos una probabilidad entre 0 y 1.

```
Input(2) → Linear(2→32) → ReLU → Linear(32→16) → ReLU → Linear(16→1) → Sigmoid → P(frena)
```

| Capa | Entrada | Salida | Activación |
|---|---|---|---|
| fc1 | 2 | 32 | ReLU |
| fc2 | 32 | 16 | ReLU |
| fc3 | 16 | 1 | Sigmoid |

> **¿Por qué Sigmoid y no Softmax?** Softmax es para clasificación multiclase. Con 2 clases (frena/no frena), Sigmoid es la elección correcta — devuelve directamente P(frena).

In [ ]:
class FrenadoMLP(nn.Module):
    """
    Red neuronal para clasificación binaria de frenado.
    Entrada: [velocidad, RPM] → Salida: probabilidad de frenado (0 a 1)
    """

    def __init__(self):
        super().__init__()
        self.fc1  = nn.Linear(2, 32)   # 2 features → 32 neuronas ocultas
        self.fc2  = nn.Linear(32, 16)  # 32 → 16 neuronas ocultas
        self.fc3  = nn.Linear(16, 1)   # 16 → 1 salida
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()    # Convierte la salida a probabilidad [0, 1]

    def forward(self, x):
        x = self.relu(self.fc1(x))     # Capa oculta 1
        x = self.relu(self.fc2(x))     # Capa oculta 2
        x = self.sigmoid(self.fc3(x))  # Salida: P(frenando)
        return x


model = FrenadoMLP().to(device)

total_params = sum(p.numel() for p in model.parameters())
print('Arquitectura del modelo:')
print(model)
print(f'\nParámetros totales: {total_params}')
print(f'  fc1: 2×32 + 32  = {2*32 + 32}')
print(f'  fc2: 32×16 + 16 = {32*16 + 16}')
print(f'  fc3: 16×1  + 1  = {16*1 + 1}')

## ⚙️ Paso 7 — Función de Pérdida y Optimizador

Para **clasificación binaria** con salida Sigmoid usamos **`BCELoss`** (Binary Cross-Entropy). Es el equivalente en PyTorch a la fórmula de entropía cruzada del script original:

$$J = -\frac{1}{m} \sum_{i=1}^{m} \left[ y_i \log(\hat{y}_i) + (1-y_i) \log(1-\hat{y}_i) \right]$$

Usamos **Adam** como optimizador, igual que en el cuadernillo `04_pytorch_save`.

In [ ]:
criterion = nn.BCELoss()                                          # Binary Cross-Entropy
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)        # Adam

print(f'Pérdida    : BCELoss  (Binary Cross-Entropy)')
print(f'Optimizador: Adam  (lr=0.001)')
print()
print('💡 BCELoss es el equivalente PyTorch de la costFunction del script original.')
print('   Ya no necesitamos calcular el gradiente a mano ni usar scipy.optimize.')

## 🏋️ Paso 8 — Bucle de Entrenamiento con DataLoader

Combinamos todo lo aprendido en los cuadernillos:

- **Mini-batches con `DataLoader`** (cuadernillo 03) — iteramos por batches en lugar de procesar todo el dataset de golpe.
- **Guardado del mejor modelo** (cuadernillo 04) — guardamos `state_dict` cuando la pérdida de validación mejora.
- El patrón de 4 pasos ya conocido: forward → loss → backward → step.

Evaluamos en el test set al final de cada época para detectar **overfitting** temprano.

In [ ]:
num_epochs   = 30
PATH_MEJOR   = './mejor_modelo_frenado.pt'   # donde guardaremos el mejor modelo

train_losses, test_losses   = [], []
train_accs,   test_accs     = [], []
mejor_loss_test = float('inf')

print('Iniciando entrenamiento con mini-batches...\n')
print(f'{"Época":>6} | {"Loss Train":>11} | {"Acc Train":>10} | {"Loss Test":>10} | {"Acc Test":>9} | Guardado')
print('-' * 75)

for epoch in range(1, num_epochs + 1):

    # ── ENTRENAMIENTO ─────────────────────────────────────────────────────────
    model.train()
    batch_losses, batch_accs = [], []

    for X_b, y_b in train_loader:           # Iteramos por BATCHES (no todo el dataset)
        X_b, y_b = X_b.to(device), y_b.to(device)

        # 1. Forward
        y_pred = model(X_b)

        # 2. Loss
        loss = criterion(y_pred, y_b)

        # 3. Backward
        optimizer.zero_grad()
        loss.backward()

        # 4. Step
        optimizer.step()

        # Métricas del batch
        batch_losses.append(loss.item())
        preds = (y_pred >= 0.5).float()
        batch_accs.append((preds == y_b).float().mean().item())

    # ── EVALUACIÓN EN TEST ────────────────────────────────────────────────────
    model.eval()
    test_batch_losses, test_batch_accs = [], []

    with torch.no_grad():
        for X_b, y_b in test_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            y_pred = model(X_b)
            loss   = criterion(y_pred, y_b)
            test_batch_losses.append(loss.item())
            preds = (y_pred >= 0.5).float()
            test_batch_accs.append((preds == y_b).float().mean().item())

    # ── GUARDAR MEJOR MODELO (cuadernillo 04) ─────────────────────────────────
    avg_train_loss = np.mean(batch_losses)
    avg_test_loss  = np.mean(test_batch_losses)
    avg_train_acc  = np.mean(batch_accs)  * 100
    avg_test_acc   = np.mean(test_batch_accs) * 100

    guardado = ''
    if avg_test_loss < mejor_loss_test:
        mejor_loss_test = avg_test_loss
        torch.save(model.state_dict(), PATH_MEJOR)   # Guardamos solo los pesos
        guardado = '✅ guardado'

    train_losses.append(avg_train_loss)
    test_losses.append(avg_test_loss)
    train_accs.append(avg_train_acc)
    test_accs.append(avg_test_acc)

    print(f'{epoch:>6} | {avg_train_loss:>11.4f} | {avg_train_acc:>9.2f}% | {avg_test_loss:>10.4f} | {avg_test_acc:>8.2f}% | {guardado}')

print(f'\n✅ Entrenamiento completado. Mejor loss test: {mejor_loss_test:.4f}')

## 📉 Paso 9 — Curvas de Aprendizaje

Graficamos **loss** y **accuracy** para train y test por separado. Esto nos permite detectar:

- **Underfitting** → ambas curvas con pérdida alta
- **Overfitting** → train mejora pero test empeora (las líneas se separan)

In [ ]:
epochs_range = range(1, num_epochs + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs_range, train_losses, lw=2, color='steelblue',  label='Train')
axes[0].plot(epochs_range, test_losses,  lw=2, color='darkorange', label='Test', linestyle='--')
axes[0].set_xlabel('Época', fontsize=11)
axes[0].set_ylabel('BCE Loss', fontsize=11)
axes[0].set_title('Curva de Pérdida', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, train_accs, lw=2, color='steelblue',  label='Train')
axes[1].plot(epochs_range, test_accs,  lw=2, color='darkorange', label='Test', linestyle='--')
axes[1].set_xlabel('Época', fontsize=11)
axes[1].set_ylabel('Accuracy (%)', fontsize=11)
axes[1].set_title('Curva de Precisión', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Curvas de Aprendizaje — MLP Frenado', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('curvas_frenado_mlp.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Accuracy final  →  Train: {train_accs[-1]:.2f}%  |  Test: {test_accs[-1]:.2f}%')

## 💾 Paso 10 — Cargar el Mejor Modelo

Siguiendo el patrón del cuadernillo `04_pytorch_save`, cargamos los pesos del mejor modelo guardado durante el entrenamiento (el que tuvo menor pérdida en test).

Esta es la forma **recomendada**: guardamos solo el `state_dict` (los pesos), no el modelo completo. Así es más eficiente y portable.

In [ ]:
# Cargamos el mejor modelo guardado durante el entrenamiento
model.load_state_dict(torch.load(PATH_MEJOR, map_location=device))
model.eval()

print(f'✅ Mejor modelo cargado desde: {PATH_MEJOR}')
print(f'   (el que tuvo menor BCE Loss en test: {mejor_loss_test:.4f})')

## 📋 Paso 11 — Evaluación Detallada

Evaluamos el modelo final con métricas más completas que el simple accuracy:

- **Precision** → de los que predijo como "frenando", ¿cuántos realmente frenaban?
- **Recall** → de los que realmente frenaban, ¿cuántos detectó correctamente?
- **F1-score** → media armónica de precision y recall
- **Matriz de confusión** → visualiza los 4 tipos de predicción

In [ ]:
# Obtenemos predicciones sobre todo el test set
all_preds, all_labels = [], []

model.eval()
with torch.no_grad():
    for X_b, y_b in test_loader:
        X_b = X_b.to(device)
        probs = model(X_b)
        preds = (probs >= 0.5).float().cpu()
        all_preds.append(preds)
        all_labels.append(y_b)

y_pred_all  = torch.cat(all_preds).numpy().flatten()
y_true_all  = torch.cat(all_labels).numpy().flatten()

print('Reporte de Clasificación (Test Set):')
print(classification_report(y_true_all, y_pred_all,
                             target_names=['No frenando (0)', 'Frenando (1)']))

In [ ]:
# Matriz de confusión visual
cm = confusion_matrix(y_true_all, y_pred_all)
labels = ['No frena\n(0)', 'Frena\n(1)']

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)

ax.set_xticks([0, 1]); ax.set_xticklabels(labels, fontsize=11)
ax.set_yticks([0, 1]); ax.set_yticklabels(labels, fontsize=11)
ax.set_xlabel('Predicción', fontsize=12)
ax.set_ylabel('Real', fontsize=12)
ax.set_title('Matriz de Confusión — Test Set', fontsize=13, fontweight='bold')

for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontsize=18, fontweight='bold', color=color)

plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'Verdaderos Negativos (TN): {tn}  — No frenaba y predijo NO FRENA ✅')
print(f'Falsos Positivos    (FP): {fp}  — No frenaba pero predijo FRENA  ❌')
print(f'Falsos Negativos    (FN): {fn}  — Frenaba pero predijo NO FRENA  ❌')
print(f'Verdaderos Positivos (TP): {tp}  — Frenaba y predijo FRENA       ✅')

## 🔮 Paso 12 — Predicción de un Momento de Carrera

Igual que el script original: dado un instante de la carrera con velocidad y RPM conocidas, el modelo nos dice si el conductor **está frenando** y con qué **probabilidad**.

> Recuerda aplicar siempre la misma normalización del entrenamiento.

In [ ]:
velocidad_test = 25.0   # m/s (aprox. 90 km/h)
rpm_test       = 8000.0 # RPM

# Normalización con los mismos mu/sigma del entrenamiento
x_nuevo      = np.array([[velocidad_test, rpm_test]], dtype=np.float32)
x_nuevo_norm = (x_nuevo - mu) / sigma
x_tensor     = torch.from_numpy(x_nuevo_norm).float().to(device)

model.eval()
with torch.no_grad():
    probabilidad = model(x_tensor).item()

prediccion = 'FRENANDO 🔴' if probabilidad >= 0.5 else 'NO FRENA 🟢'

print('=' * 52)
print('   PREDICCIÓN — MOMENTO DE CARRERA')
print('=' * 52)
print(f'  Velocidad  : {velocidad_test} m/s  ({velocidad_test*3.6:.1f} km/h)')
print(f'  RPM Motor  : {rpm_test:.0f}')
print('-' * 52)
print(f'  P(frenando): {probabilidad*100:.2f}%')
print(f'  Predicción : {prediccion}')
print('=' * 52)

## 🧪 Bonus — Prueba con tus propios datos de telemetría

Cambia los valores de velocidad y RPM y vuelve a ejecutar la celda.

In [ ]:
# ✏️ Modifica estos valores
mi_velocidad = 60.0    # m/s — velocidad alta en recta
mis_rpm      = 4000.0  # RPM bajas pueden indicar frenada o curva lenta

# --- No modificar debajo ---
x_mi = np.array([[mi_velocidad, mis_rpm]], dtype=np.float32)
x_mi_norm = (x_mi - mu) / sigma
x_mi_t = torch.from_numpy(x_mi_norm).float().to(device)

model.eval()
with torch.no_grad():
    prob = model(x_mi_t).item()

estado = 'FRENANDO 🔴' if prob >= 0.5 else 'NO FRENA 🟢'
print(f'{mi_velocidad} m/s | {mis_rpm:.0f} RPM  →  P(frena)={prob*100:.1f}%  →  {estado}')

## 📋 Resumen — Script original vs Este notebook

| Aspecto | Script original (NumPy + scipy) | Este notebook (PyTorch) |
|---|---|---|
| Modelo | Regresión logística (1 capa) | MLP 2 capas ocultas + ReLU |
| Optimización | `scipy.optimize.minimize` (TNC) | Adam + `autograd` |
| Datos | Arrays NumPy directos | `Dataset` + `DataLoader` (mini-batches) |
| Train/Test split | ❌ (solo train) | ✅ 80/20 estratificado |
| Guardado | ❌ | ✅ `state_dict` del mejor modelo |
| Evaluación | Solo accuracy | Accuracy + Precision + Recall + F1 + Confusión |
| GPU | ❌ | ✅ `.to(device)` automático |

---

### ¿Qué sigue?

- 🔄 Exportar con **TorchScript** para producción: `torch.jit.script(model)`
- 📦 Añadir más features de telemetría: aceleración lateral, posición en pista
- ⚖️ Manejar el **desbalance de clases** con `pos_weight` en `BCEWithLogitsLoss`
- 🧪 Probar **Dropout** (`nn.Dropout(0.3)`) si hay overfitting